In [4]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
data = pd.read_csv('HappinessSurvey2020.csv')

data.drop(columns=['Y'])
Y = data['Y']
X, y = make_classification(
    n_samples=1200, 
    n_features=20, 
    n_informative=15, 
    n_classes=2, 
    random_state=42
)

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split( X, y, random_state=42)

# 2. Evaluate Baseline Model (Default Parameters)
baseline_rf = RandomForestClassifier(random_state=42)
baseline_rf.fit(X_train, y_train)
baseline_preds = baseline_rf.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_preds)

print(f"Baseline Accuracy (Default Params): {baseline_accuracy:.4f}")

# 3. Define the Hyperparameter Grid for Tuning
# We focus on the parameters that influence accuracy and overfitting the most:
param_grid = {
    'n_estimators': [50, 100, 200],       # Number of decision trees
    'max_depth': [None, 10, 20, 30],       # Maximum depth of the trees
    'min_samples_split': [2, 5, 10],       # Minimum samples needed to split a node
    'min_samples_leaf': [1, 2, 4],         # Minimum samples required at a leaf node
    'max_features': ['sqrt', 'log2']       # Number of features to consider at each split
}

# Initialize the Random Forest Base model
rf = RandomForestClassifier(random_state=42)

# 4. Use GridSearchCV to search for the best combinations using 5-fold cross-validation
# n_jobs=-1 uses all available processor cores to speed up training
grid_search = GridSearchCV(
    estimator=rf, 
    param_grid=param_grid, 
    cv=5, 
    scoring='accuracy', 
    n_jobs=-1, 
    verbose=1
)

# Run the parameter tuning process
grid_search.fit(X_train, y_train)

# 5. Extract Best Parameters and Calculate the New Accuracy
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_

# Predict on test data using the tuned model
tuned_preds = best_model.predict(X_test)
tuned_accuracy = accuracy_score(y_test, tuned_preds)

print("\n--- Tuning Results ---")
print(f"Best Hyperparameters Found: {best_params}")
print(f"Optimized Accuracy: {tuned_accuracy:.4f}")
print(f"Accuracy Improvement: {(tuned_accuracy - baseline_accuracy) * 100:+.2f}%")

Baseline Accuracy (Default Params): 0.9000
Fitting 5 folds for each of 216 candidates, totalling 1080 fits

--- Tuning Results ---
Best Hyperparameters Found: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 50}
Optimized Accuracy: 0.8833
Accuracy Improvement: -1.67%
